In [ ]:
"""
Python equivalent of the MATLAB ``example_zFilter`` script.

This script tests the computation of z-filters and verifies the results
against a numerical differentiation method.
"""
from __future__ import annotations

from pathlib import Path
import sys
import numpy as np
from scipy.signal import freqz, sosfreqz
import matplotlib.pyplot as plt

from pyFDN.auxiliary.zsos import ZSOS
from pyFDN.auxiliary.ztf import ZTF
from pyFDN.generate.is_almost_zero import is_almost_zero

def plot_lines(w, h, hh, dh, dhh, title_suffix=""):
    """Helper function to plot frequency and derivative responses."""
    plt.figure()
    plt.plot(w, np.real(h), label='Real Freqz')
    plt.plot(w, np.imag(h), label='Imag Freqz')
    offset = 0.1
    plt.plot(w, np.real(hh) + offset, label='Real zFilter', linestyle='--')
    plt.plot(w, np.imag(hh) + offset, label='Imag zFilter', linestyle='--')
    plt.grid(True)
    plt.legend()
    plt.xlabel('Frequency [rad]')
    plt.ylabel('Amplitude [lin]')
    plt.title(f'Frequency Response {title_suffix}')

    plt.figure()
    plt.plot(w[1:], np.real(dh), label='Real Freqz Derivative')
    plt.plot(w[1:], np.imag(dh), label='Imag Freqz Derivative')
    offset = 0.1
    plt.plot(w, np.real(dhh) + offset, label='Real zFilter Derivative', linestyle='--')
    plt.plot(w, np.imag(dhh) + offset, label='Imag zFilter Derivative', linestyle='--')
    plt.grid(True)
    plt.legend()
    plt.xlabel('Frequency [rad]')
    plt.ylabel('Amplitude [lin]')
    plt.title(f'Derivative Response {title_suffix}')


def probe_zfilter(z_filter, ww):
    """Evaluate filter and its derivative across a range of frequencies."""
    hh = np.zeros_like(ww, dtype=np.complex128)
    dhh = np.zeros_like(ww, dtype=np.complex128)
    
    for i, w_val in enumerate(ww):
        mat_at = z_filter.at(w_val)
        hh[i] = mat_at[0, 0] if mat_at.ndim == 2 else mat_at[0]
        
        mat_der = z_filter.der(w_val)
        dhh[i] = mat_der[0, 0] if mat_der.ndim == 2 else mat_der[0]
    
    return hh, dhh


def main():
    """Main function that runs both tests."""
    print("Example for zFilter")
    print("Testing script for the computation of zFilters and verification with a")
    print("numerical differentiation method.")
    print()
    
    # Set random seed for reproducibility
    np.random.seed(2)
    nfft = 2**14
    
    print("=== Test: Transfer function and derivative ===")
    n, m, order = 2, 1, 5
    a = np.random.randn(n, m, order)
    b = np.random.randn(n, m, order)
    
    # Frequency response and numerical derivative
    b_1d = b[0, 0, :].squeeze()
    a_1d = a[0, 0, :].squeeze()
    w, h = freqz(b_1d, a_1d, worN=nfft)
    ww = np.exp(1j * w)
    dh = np.diff(h) / np.diff(ww)
    
    # Same with zFilter object
    z_filter = ZTF(b, a, is_diagonal=True)
    hh, dhh = probe_zfilter(z_filter, ww)
    
    # Plot results
    plot_lines(w, h, hh, dh, dhh, "(ZTF)")
    
    # Assertions
    print(f"ZTF frequency response max error: {np.max(np.abs(h - hh))}")
    print(f"ZTF derivative max error: {np.max(np.abs(dh - dhh[1:]))}")
    
    assert is_almost_zero(h - hh), "ZTF frequency response mismatch"
    assert is_almost_zero(dh - dhh[1:], tol=0.15), "ZTF derivative mismatch"
    print("ZTF tests passed!")
    print()
    
    print("=== SOS Test ===")
    plt.close('all')
    nsos = 3  # at least 2
    n, m = 2, 1
    sos = np.random.rand(n, m, nsos, 6)
    
    # Normalize SOS format for scipy (first denominator coefficient should be 1)
    for i in range(nsos):
        a0 = sos[0, 0, i, 3]
        if a0 != 0:
            sos[0, 0, i, :3] /= a0  # normalize numerator
            sos[0, 0, i, 3:] /= a0  # normalize denominator
    
    # Frequency response and numerical derivative
    sos_1d = sos[0, 0, :, :].squeeze()
    w, h = sosfreqz(sos_1d, worN=nfft)
    ww = np.exp(1j * w)
    dh = np.diff(h) / np.diff(ww)
    
    # Same with zFilter object
    z_filter = ZSOS(sos, is_diagonal=False)
    hh, dhh = probe_zfilter(z_filter, ww)
    
    # Plot results
    plot_lines(w, h, hh, dh, dhh, "(ZSOS)")
    
    # Assertions
    print(f"ZSOS frequency response max error: {np.max(np.abs(h - hh))}")
    print(f"ZSOS derivative max error: {np.max(np.abs(dh - dhh[1:]))}")
    
    assert is_almost_zero(h - hh), "ZSOS frequency response mismatch"
    assert is_almost_zero(dh - dhh[1:], tol=0.15), "ZSOS derivative mismatch"
    print("ZSOS tests passed!")
    
    # Show all plots
    plt.show()


if __name__ == "__main__":
    main()